# LAB | Hyperparameter Tuning

**Load the data**

Finally step in order to maximize the performance on your Spaceship Titanic model.

The data can be found here:

https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv

Metadata

https://github.com/data-bootcamp-v4/data/blob/main/spaceship_titanic.md

So far we've been training and evaluating models with default values for hyperparameters.

Today we will perform the same feature engineering as before, and then compare the best working models you got so far, but now fine tuning it's hyperparameters.

In [ ]:
#Libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

In [ ]:
spaceship = pd.read_csv("https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv")
spaceship.head()

Now perform the same as before:
- Feature Scaling
- Feature Selection


In [ ]:
spaceship.shape

In [ ]:
spaceship.dtypes

In [ ]:
spaceship.isna().sum()

In [ ]:
spaceship = spaceship.dropna()
spaceship = spaceship.reset_index(drop=True)

In [ ]:
spaceship['Cabin'] = spaceship['Cabin'].apply(lambda x: x[0])
spaceship.Cabin.value_counts()

In [ ]:
spaceship = spaceship.drop(["PassengerId", "Name"], axis=1)
spaceship

In [ ]:
features = spaceship.drop(columns = "Transported")
target = spaceship["Transported"]

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(spaceship, target, test_size = 0.20, random_state=0)

In [ ]:
from sklearn.preprocessing import OneHotEncoder

ohe = OneHotEncoder(sparse_output=False)

ohe.fit(X_train[['HomePlanet', 'CryoSleep', 'Cabin', 'Destination', 'VIP']])

In [ ]:
X_train_ohe = ohe.transform(X_train[['HomePlanet', 'CryoSleep', 'Cabin', 'Destination', 'VIP']])
X_test_ohe = ohe.transform(X_test[['HomePlanet', 'CryoSleep', 'Cabin', 'Destination', 'VIP']])
X_train_ohe

In [ ]:
X_train_ohe_df = pd.DataFrame(X_train_ohe, columns=ohe.get_feature_names_out(), index=X_train.index)
X_train_ohe_df

In [ ]:
X_test_ohe_df = pd.DataFrame(X_test_ohe, columns=ohe.get_feature_names_out(), index=X_test.index)
X_test_ohe_df

In [ ]:
X_train_num = X_train.select_dtypes(include=['number'])
X_train = pd.concat([X_train_num, X_train_ohe_df], axis = 1)
X_train

In [ ]:
X_test_num = X_test.select_dtypes(include=['number'])
X_test = pd.concat([X_test_num, X_test_ohe_df], axis = 1)
X_test

In [ ]:
from sklearn.preprocessing import MinMaxScaler, StandardScaler

normalizer = MinMaxScaler()
normalizer.fit(X_train)

In [ ]:
X_train_norm = normalizer.transform(X_train)
X_train_norm = pd.DataFrame(X_train_norm, columns=X_train.columns, index=X_train.index )
X_train_norm.head()

In [ ]:
X_test_norm = normalizer.transform(X_test)
X_test_norm = pd.DataFrame(X_test_norm, columns=X_test.columns, index=X_test.index )
X_test_norm.head()

- Now let's use the best model we got so far in order to see how it can improve when we fine tune it's hyperparameters.

In [ ]:
from sklearn.ensemble import AdaBoostClassifier
ab_clf = AdaBoostClassifier(n_estimators=50)
ab_clf.fit(X_train_norm, y_train)

- Evaluate your model

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
y_pred_test_ab = ab_clf.predict(X_test_norm)

print(f"Accuracy: {accuracy_score(y_test, y_pred_test_ab):.2f}")
print(f"Precision: {precision_score(y_test, y_pred_test_ab):.2f}")
print(f"Recall: {recall_score(y_test, y_pred_test_ab):.2f}")
print(f"F1 Score: {f1_score(y_test, y_pred_test_ab):.2f}")

**Grid/Random Search**

For this lab we will use Grid Search.

- Define hyperparameters to fine tune.

In [ ]:
import optuna
import optuna.visualization as vis
import time
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

In [ ]:
parameter_grid = {'n_estimators': [50, 100], 'learning_rate': [0.01, 0.1, 1]}

- Run Grid Search

In [ ]:
gs = GridSearchCV(ab_clf, param_grid=parameter_grid, cv=10, scoring='accuracy', verbose=10)

# Run the grid search
start_time = time.time()
gs.fit(X_train, y_train)
end_time = time.time()

print(f"Time taken to find the best combination of hyperparameters among the given ones: {end_time - start_time:.4f} seconds")

print(f"The best combination of hyperparameters has been: {gs.best_params_}")
print(f"The best cross-validated accuracy is: {gs.best_score_: .4f}")

# Extract the best model
best_ada_clf = gs.best_estimator_

# Evaluate on the test set
y_pred_test_ada = best_ada_clf.predict(X_test)

# Calculate metrics for the classifier
accuracy_ada = accuracy_score(y_test, y_pred_test_ada)
precision_ada = precision_score(y_test, y_pred_test_ada)
recall_ada = recall_score(y_test, y_pred_test_ada)
f1_ada = f1_score(y_test, y_pred_test_ada)

print("Evaluation for AdaBoost Classifier:")
print(f"Accuracy: {accuracy_ada:.2f}")
print(f"Precision: {precision_ada:.2f}")
print(f"Recall: {recall_ada:.2f}")
print(f"F1 Score: {f1_ada:.2f}")

- Evaluate your model

In [ ]:
#The best combination of hyperparameters is 50 n_estimators and learning_rate of 1. Since this was the default setting of the AdaBoost when fitting the model earlier, the performnace metrics stayed the same.